# Random Forest (Default Params) — F1 Race Position Prediction

Trains a `RandomForestRegressor` with default hyperparameters and evaluates it with **temporal cross-validation** — an expanding-window walk-forward split over seasons, so every fold is only ever validated on a season that comes *after* the seasons it was trained on. This mirrors the real deployment setting (predict an upcoming season using only past seasons) and avoids the leakage a random/shuffled K-fold split would introduce.

Metrics (same as the baseline notebook, for direct comparison):
- **MAE** (Mean Absolute Error) — average position error
- **Spearman ρ** — rank-order correlation between predicted and actual finishing positions
- **Macro F1** — treats each position (1–20) as a class; averages F1 equally across all positions regardless of frequency

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, f1_score
from scipy.stats import spearmanr

sys.path.insert(0, str(Path.cwd().parent.parent))
from pipeline.feature_engineering.feature_engineering import FINAL_FEATURES, TARGET

## 2. Load Data

Train/test splits were produced by the feature engineering pipeline and persisted as Parquet files under `data/`. `train.parquet` holds seasons 2019–2024; `test.parquet` holds the held-out 2025 season.

In [2]:
train = pd.read_parquet('../../data/train.parquet')
test = pd.read_parquet('../../data/test.parquet')

print(f"train: {train.shape[0]} rows, years {sorted(train['year'].unique())}")
print(f"test:  {test.shape[0]} rows, years {sorted(test['year'].unique())}")

train: 2556 rows, years [np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]
test:  479 rows, years [np.int32(2025)]


## 3. Prepare Features and Target

`RacePosition` (1–20) is the regression target. Model inputs are restricted to `FINAL_FEATURES` — the locked feature list from the feature engineering pipeline — which excludes identifier columns (`DriverId`, `DriverNumber`, `year`) that carry no predictive signal on their own.

In [3]:
train_X = train[FINAL_FEATURES]
train_Y = train[TARGET]
test_X = test[FINAL_FEATURES]
test_Y = test[TARGET]

## 4. Validate Data

Confirm no missing values before training — the feature engineering pipeline should have handled imputation upstream.

In [4]:
for name, df in [("train_X", train_X), ("train_Y", train_Y), ("test_X", test_X), ("test_Y", test_Y)]:
    n_null = df.isna().sum().sum() if hasattr(df, "columns") else df.isna().sum()
    status = "OK" if n_null == 0 else f"WARNING: {n_null} nulls"
    print(f"{name}: {status}")

train_X: OK
train_Y: OK
test_X: OK
test_Y: OK


## 5. Metric Helpers

Same definitions as the baseline notebook, so scores are directly comparable.

In [5]:
def macro_f1(y_true, y_pred):
    """Round continuous predictions to nearest integer position, then compute macro F1."""
    y_pred_int = pd.Series(y_pred).round().clip(1, 20).astype(int)
    return f1_score(y_true.astype(int), y_pred_int, average="macro", zero_division=0)

def safe_spearmanr(y_true, y_pred):
    """Return 0.0 when predictions are constant (ρ is undefined for constant input)."""
    if pd.Series(y_pred).nunique() == 1:
        return 0.0
    return spearmanr(y_true, y_pred).statistic

def score(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Spearman_rho": safe_spearmanr(y_true, y_pred),
        "Macro_F1": macro_f1(y_true, y_pred),
    }

## 6. Temporal Cross-Validation

Expanding-window walk-forward split over the 6 training seasons (2019–2024): fold *i* trains on every season strictly before the validation season and validates on the next one. This yields 5 folds:

| Fold | Train years | Validation year |
|---|---|---|
| 1 | 2019 | 2020 |
| 2 | 2019–2020 | 2021 |
| 3 | 2019–2021 | 2022 |
| 4 | 2019–2022 | 2023 |
| 5 | 2019–2023 | 2024 |

`RandomForestRegressor` is trained with default hyperparameters (only `random_state` is fixed, for reproducibility) in every fold.

In [6]:
years = sorted(train['year'].unique())
cv_rows = []

for i in range(1, len(years)):
    train_years = years[:i]
    val_year = years[i]

    fold_train_mask = train['year'].isin(train_years)
    fold_val_mask = train['year'] == val_year

    fold_model = RandomForestRegressor(random_state=42)
    fold_model.fit(train_X[fold_train_mask], train_Y[fold_train_mask])
    fold_preds = fold_model.predict(train_X[fold_val_mask])

    metrics = score(train_Y[fold_val_mask], fold_preds)
    metrics["fold"] = i
    metrics["train_years"] = f"{train_years[0]}-{train_years[-1]}" if len(train_years) > 1 else str(train_years[0])
    metrics["val_year"] = val_year
    cv_rows.append(metrics)

cv_results = pd.DataFrame(cv_rows)[["fold", "train_years", "val_year", "MAE", "Spearman_rho", "Macro_F1"]]
cv_results

,fold,train_years,val_year,MAE,Spearman_rho,Macro_F1
0,1,2019,2020,3.778676,0.565192,0.054643
1,2,2019-2020,2021,3.455581,0.644766,0.079248
2,3,2019-2021,2022,3.733622,0.550363,0.067942
3,4,2019-2022,2023,3.567153,0.601617,0.079010
4,5,2019-2023,2024,3.175866,0.700236,0.079972


## 7. CV Summary

In [7]:
cv_summary = cv_results[["MAE", "Spearman_rho", "Macro_F1"]].agg(["mean", "std"])
print(f"{'Metric':<14} {'Mean':>8}  {'Std':>8}")
print("-" * 34)
for col in ["MAE", "Spearman_rho", "Macro_F1"]:
    print(f"{col:<14} {cv_summary.loc['mean', col]:>8.3f}  {cv_summary.loc['std', col]:>8.3f}")

Metric             Mean       Std
----------------------------------
MAE               3.542     0.242
Spearman_rho      0.612     0.061
Macro_F1          0.072     0.011


## 8. Final Model — Evaluate on Held-Out 2025 Test Set

Refit on the full training set (2019–2024) and score once on the untouched 2025 test season, alongside the baselines from `reports/baseline_results.csv` for comparison.

In [8]:
final_model = RandomForestRegressor(random_state=42)
final_model.fit(train_X, train_Y)
test_preds = final_model.predict(test_X)

test_results = score(test_Y, test_preds)

baseline_results = pd.read_csv("../../reports/baseline_results.csv").set_index("model")
comparison = pd.concat([
    baseline_results,
    pd.DataFrame({"RandomForest": test_results}).T.rename_axis("model"),
])

print(f"{'Model':<20} {'MAE':>6}  {'Spearman ρ':>10}  {'Macro F1':>8}")
print("-" * 52)
for model, metrics in comparison.iterrows():
    print(f"{model:<20} {metrics['MAE']:>6.2f}  {metrics['Spearman_rho']:>10.3f}  {metrics['Macro_F1']:>8.3f}")

Model                   MAE  Spearman ρ  Macro F1
----------------------------------------------------
GridPosition          10.44       0.652     0.005
DummyRegressor         4.99       0.000     0.005
RandomForest           3.53       0.617     0.077


## 9. Train vs CV Error — Overfitting Check

Compare error on the data `final_model` was fit on (train, in-sample) against the mean CV error (out-of-sample, Section 7) and the held-out 2025 test error (Section 8). A large train-vs-CV/test gap signals overfitting — a real risk here since the model above uses unrestricted depth / unlimited boosting rounds by default.

In [9]:
train_preds = final_model.predict(train_X)
train_results = score(train_Y, train_preds)

error_comparison = pd.DataFrame({
    "Train (in-sample)": train_results,
    "CV (mean, out-of-sample)": cv_summary.loc["mean"],
    "Test (2025, held-out)": test_results,
}).T[["MAE", "Spearman_rho", "Macro_F1"]]

error_comparison

,MAE,Spearman_rho,Macro_F1
Train (in-sample),1.259965,0.971881,0.239146
"CV (mean, out-of-sample)",3.542180,0.612435,0.072163
"Test (2025, held-out)",3.531587,0.616917,0.077124


## 10. Feature Importance

First-pass feature importance from `final_model` (fit on the full 2019–2024 training set in Section 8). Importance is impurity-based (mean decrease in variance, i.e. Gini importance) — biased toward high-cardinality numeric features, so treat the ranking as a rough first pass, not a causal signal.

In [10]:
feature_importance = pd.Series(
    final_model.feature_importances_, index=FINAL_FEATURES, name="importance"
).sort_values(ascending=False)

feature_importance

GridPosition                   0.300553
TeamFinish_ewm                 0.233309
DriverFinish_ewm               0.108846
LapStd_lag1                    0.077580
TeamFinish_roll3_inseason      0.058407
DriverFinish_roll3_inseason    0.051297
round_number                   0.046740
Meeting.Circuit.ShortName      0.045519
DriverFinish_lag1              0.043931
TeamName                       0.033818
Name: importance, dtype: float64

## 10a. Multicollinearity Check (VIF)

Variance Inflation Factor (VIF) measures how much a feature's variance is inflated by linear correlation with the other features — VIF > 5 is the usual "moderately collinear" threshold. Computed on the 8 continuous/rolling features only; `TeamName` and `Meeting.Circuit.ShortName` are label-encoded categoricals with an arbitrary numeric ordering, so a linear-redundancy statistic isn't meaningful for them — they're excluded here and kept regardless of VIF.

In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

CATEGORICAL_FEATURES = ["TeamName", "Meeting.Circuit.ShortName"]
vif_features = [f for f in FINAL_FEATURES if f not in CATEGORICAL_FEATURES]

vif_X = add_constant(train_X[vif_features].dropna())
vif = pd.Series(
    [variance_inflation_factor(vif_X.values, i) for i in range(vif_X.shape[1])],
    index=vif_X.columns,
    name="VIF",
).drop("const").sort_values(ascending=False)

vif

TeamFinish_ewm                 25.175282
DriverFinish_ewm               25.139299
TeamFinish_roll3_inseason      22.441007
DriverFinish_roll3_inseason    18.338941
DriverFinish_lag1               3.513101
GridPosition                    1.711442
round_number                    1.002871
LapStd_lag1                     1.001978
Name: VIF, dtype: float64

## 10b. Feature Selection — Drop Near-Zero-Importance, High-VIF Features

Two independent signals both have to flag a feature before it's dropped:
- **Near-zero importance** — impurity importance share below `1 / 10` features (10%), i.e. contributing less than a uniformly-weighted feature would.
- **High VIF** — VIF > 5 (Section 10a).

A feature that's collinear but still pulling its weight (e.g. `TeamFinish_ewm` / `DriverFinish_ewm` — both high VIF *and* top-3 importance) is kept; only the redundant, low-importance side of a collinear pair gets cut.

In [12]:
IMPORTANCE_THRESHOLD = 1 / len(FINAL_FEATURES)  # 10% — a uniformly-weighted feature's share
VIF_THRESHOLD = 5

importance_share = feature_importance / feature_importance.sum()

selection = pd.DataFrame({
    "importance_share": importance_share,
    "VIF": vif.reindex(FINAL_FEATURES),
})
selection["near_zero_importance"] = selection["importance_share"] < IMPORTANCE_THRESHOLD
selection["high_vif"] = selection["VIF"] > VIF_THRESHOLD
selection["drop"] = selection["near_zero_importance"] & selection["high_vif"].fillna(False)

SELECTED_FEATURES = selection[~selection["drop"]].index.tolist()
dropped_features = selection[selection["drop"]].index.tolist()

print(f"Dropped ({len(dropped_features)}): {dropped_features}")
print(f"Retained ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}\n")

selection.sort_values("importance_share", ascending=False)

Dropped (2): ['DriverFinish_roll3_inseason', 'TeamFinish_roll3_inseason']
Retained (8): ['DriverFinish_ewm', 'DriverFinish_lag1', 'GridPosition', 'LapStd_lag1', 'Meeting.Circuit.ShortName', 'TeamFinish_ewm', 'TeamName', 'round_number']



,importance_share,VIF,near_zero_importance,high_vif,drop
GridPosition,0.300553,1.711442,False,False,False
TeamFinish_ewm,0.233309,25.175282,False,True,False
DriverFinish_ewm,0.108846,25.139299,False,True,False
LapStd_lag1,0.077580,1.001978,True,False,False
TeamFinish_roll3_inseason,0.058407,22.441007,True,True,True
DriverFinish_roll3_inseason,0.051297,18.338941,True,True,True
round_number,0.046740,1.002871,True,False,False
Meeting.Circuit.ShortName,0.045519,NaN,True,False,False
DriverFinish_lag1,0.043931,3.513101,True,False,False
TeamName,0.033818,NaN,True,False,False


## 10c. Refit and Compare with Reduced Feature Set

Rerun the same temporal CV (Section 6) and held-out 2025 evaluation (Section 8), restricted to `SELECTED_FEATURES`, and compare against the all-features model above.

In [13]:
def run_cv(feature_cols):
    rows = []
    for i in range(1, len(years)):
        cv_train_years = years[:i]
        val_year = years[i]

        fold_train_mask = train['year'].isin(cv_train_years)
        fold_val_mask = train['year'] == val_year

        fold_model = RandomForestRegressor(random_state=42)
        fold_model.fit(train[feature_cols][fold_train_mask], train_Y[fold_train_mask])
        fold_preds = fold_model.predict(train[feature_cols][fold_val_mask])

        metrics = score(train_Y[fold_val_mask], fold_preds)
        metrics["fold"] = i
        metrics["train_years"] = f"{cv_train_years[0]}-{cv_train_years[-1]}" if len(cv_train_years) > 1 else str(cv_train_years[0])
        metrics["val_year"] = val_year
        rows.append(metrics)
    return pd.DataFrame(rows)[["fold", "train_years", "val_year", "MAE", "Spearman_rho", "Macro_F1"]]

reduced_cv_results = run_cv(SELECTED_FEATURES)
reduced_cv_summary = reduced_cv_results[["MAE", "Spearman_rho", "Macro_F1"]].agg(["mean", "std"])

reduced_model = RandomForestRegressor(random_state=42)
reduced_model.fit(train[SELECTED_FEATURES], train_Y)
reduced_test_results = score(test_Y, reduced_model.predict(test[SELECTED_FEATURES]))

feature_selection_comparison = pd.DataFrame({
    "All features (CV mean)": cv_summary.loc["mean"],
    "Reduced features (CV mean)": reduced_cv_summary.loc["mean"],
    "All features (2025 test)": pd.Series(test_results),
    "Reduced features (2025 test)": pd.Series(reduced_test_results),
}).T[["MAE", "Spearman_rho", "Macro_F1"]]

feature_selection_comparison

,MAE,Spearman_rho,Macro_F1
All features (CV mean),3.542180,0.612435,0.072163
Reduced features (CV mean),3.547173,0.612200,0.076613
All features (2025 test),3.531587,0.616917,0.077124
Reduced features (2025 test),3.515595,0.614057,0.079802


## 11. Save Results

Persists the reduced-feature-set CV results and test comparison (Section 10c) — the model actually selected after the VIF/importance check — not the diagnostic all-10-feature run from Sections 6–8.

In [14]:
final_comparison = pd.concat([
    baseline_results,
    pd.DataFrame({"RandomForest": reduced_test_results}).T.rename_axis("model"),
])

reduced_cv_results.to_csv("../../reports/random_forest_cv_results.csv", index=False)
final_comparison.reset_index().to_csv("../../reports/random_forest_test_results.csv", index=False)
final_comparison

,MAE,Spearman_rho,Macro_F1
model,,,
GridPosition,10.437773,0.651649,0.005263
DummyRegressor,4.990574,0.000000,0.004771
RandomForest,3.515595,0.614057,0.079802


## 12. Takeaways

- The temporal CV folds show how default-parameter Random Forest performance evolves as more historical seasons become available for training — early folds (fold 1: train on a single season) are a much weaker test than later folds.
- The final row compares the model, fit on all six training seasons, against the `GridPosition` and `DummyRegressor` baselines on the untouched 2025 season. A meaningful model should beat `GridPosition` on Spearman ρ and Macro F1 while staying competitive on MAE.
- **Overfitting is severe with default params**: train MAE is 1.26 vs. 3.54 on CV and 3.53 on test — a ~2.3-point gap (train Spearman ρ 0.97 vs. CV 0.61). Unbounded tree depth lets the forest memorize the training seasons; `max_depth`, `min_samples_leaf`, and `max_features` are the highest-priority regularization targets, ahead of `n_estimators`.
- **Feature importance (impurity-based) is concentrated**: on the diagnostic all-10-feature fit, `GridPosition` alone accounts for ~30% of importance, followed by `TeamFinish_ewm` (~23%) and `DriverFinish_ewm` (~11%) — the model leans heavily on qualifying position plus recent team/driver form, with the remaining seven features splitting the rest.
- **Multicollinearity check drops 2 redundant features (Sections 10a–10c)**: `TeamFinish_ewm`/`TeamFinish_roll3_inseason` and `DriverFinish_ewm`/`DriverFinish_roll3_inseason` are highly collinear (VIF 18–25 — the EWM and in-season rolling mean are both smoothed versions of the same recent-form signal). Within each pair, the `_roll3_inseason` feature carries the least importance (5.8% and 5.1%, both below the 10% near-zero-importance bar), so both are dropped and the model refit on the remaining 8 `SELECTED_FEATURES`. CV MAE moves from 3.542 to 3.547 and test MAE from 3.532 to 3.516 — well inside the ~0.24 fold-to-fold CV std, i.e. no real regression. `random_forest_cv_results.csv` / `random_forest_test_results.csv` now reflect this reduced 8-feature model.
